In [1]:
# ── SEKCJA 1 (kwartalna): ETL — wczytanie i przygotowanie danych ──────────────

import pandas as pd
import requests
import numpy as np
from pathlib import Path
import gc

# Konfiguracja ścieżek względnych (dla GitHub)
PROJECT_ROOT = Path.cwd().parent
DATA_RAW       = PROJECT_ROOT / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'processed'

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# ══════════════════════════════════════════════════════════════════════════════
# 1.1 PKB KWARTALNE — Eurostat API
# Tablica:   namq_10_gdp
# Parametry: geo=PL, na_item=B1GQ, s_adj=SCA, unit=CP_MNAC, freq=Q
# Zakres:    1999-Q1 – 2025-Q4
# ══════════════════════════════════════════════════════════════════════════════

print('⏳ Pobieranie danych PKB z Eurostat API...')

url_pkb = (
    "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"
    "namq_10_gdp"
    "?freq=Q"
    "&unit=CP_MNAC"
    "&na_item=B1GQ"
    "&s_adj=SCA"
    "&geo=PL"
    "&sinceTimePeriod=1999-Q1"
    "&untilTimePeriod=2025-Q4"
)

response    = requests.get(url_pkb)
response.raise_for_status()
data_pkb    = response.json()
time_labels = data_pkb["dimension"]["time"]["category"]["label"]
values_pkb  = data_pkb["value"]

rows_pkb = []
for idx, value in values_pkb.items():
    kwartal = list(time_labels.values())[int(idx)]
    kwartal = kwartal.replace('-', '')   # '2025-Q1' → '2025Q1'
    rows_pkb.append({"kwartal": kwartal, "pkb_mln_pln": value})

pkb_q = (
    pd.DataFrame(rows_pkb)
    .sort_values("kwartal")
    .set_index("kwartal")
)
pkb_q['pkb_mld'] = pkb_q['pkb_mln_pln'] / 1000
pkb_q = pkb_q.drop(columns=['pkb_mln_pln'])

print(f'✅ PKB kwartalne: {pkb_q.index.min()}–{pkb_q.index.max()}, '
      f'n={len(pkb_q)}')

# ══════════════════════════════════════════════════════════════════════════════
# 1.2 KONFIGURACJA FORMATÓW PLIKÓW
#
# Pięć formatów plików sprawozdań operatywnych MF:
#
# FORMAT A — rok 1999
#   kol_nazwa=1, kol_suma=8
#
# FORMAT B — lata 2000–2003
#   kol_nazwa=2, kol_suma=10
#
# FORMAT C — lata 2004–2015
#   kol_nazwa=3, kol_suma=11
#
# FORMAT D — lata 2016–2017
#   kol_nazwa=1, kol_suma=9
#   Uwaga: plik 2017 ma rozszerzenie .XLS (wielkie litery)
#
# FORMAT E — lata 2018–2024 (nowy format)
#   kol_nazwa=0, kol_suma=7
#   4 osobne bloki narastające w tej samej kolumnie
#
# We wszystkich formatach:
#   Akcyza fraza: 'Podatek akcyzowy' (wielka P — wyklucza podpozycje)
#   Wartości w tys. PLN → / 1_000_000 → mld PLN
# ══════════════════════════════════════════════════════════════════════════════

FORMAT_MAP = {
    1999:         (1, 8),
    **{r: (2, 10) for r in range(2000, 2004)},
    **{r: (3, 11) for r in range(2004, 2016)},
    2016:         (1, 9),
    2017:         (1, 9),
}

PODATKI_Q = {
    'towarów i usług':  'vat',
    'osób prawnych':    'cit',
    'Podatek akcyzowy': 'akcyza',
}

KOL_NOWY = 7


def wykryj_format_e(df_raw: pd.DataFrame) -> bool:
    return df_raw[0].astype(str).str.contains(
        'towarów i usług', case=False, na=False
    ).any()


def parsuj_bloki(df_raw: pd.DataFrame, kol_nazwa: int,
                 kol_suma: int, rok: int) -> dict:
    df_raw[kol_nazwa] = df_raw[kol_nazwa].astype(str).str.strip()
    df_raw[kol_suma]  = pd.to_numeric(df_raw[kol_suma], errors='coerce')

    dane = {}
    for fraza, skrot in PODATKI_Q.items():
        maska  = df_raw[kol_nazwa].str.contains(fraza, case=False, na=False)
        wyniki = df_raw[maska]

        if len(wyniki) < 4:
            print(f'  ⚠️  [{rok}] "{fraza}": '
                  f'{len(wyniki)} trafień zamiast 4 — NaN')
            dane[skrot] = [float('nan')] * 4
            continue

        vals = [pd.to_numeric(wyniki.iloc[i][kol_suma], errors='coerce')
                for i in range(4)]
        kw   = [vals[0], vals[1]-vals[0], vals[2]-vals[1], vals[3]-vals[2]]
        dane[skrot] = [
            v / 1_000_000 if pd.notna(v) else float('nan')
            for v in kw
        ]

    return dane


def parsuj_format_e(df_raw: pd.DataFrame, rok: int) -> dict:
    df_raw[0]       = df_raw[0].astype(str).str.strip()
    df_raw[KOL_NOWY] = pd.to_numeric(df_raw[KOL_NOWY], errors='coerce')

    dane = {}
    for fraza, skrot in PODATKI_Q.items():
        case_sensitive = fraza.startswith('Podatek')
        maska  = df_raw[0].str.contains(fraza, case=case_sensitive, na=False)
        wyniki = df_raw[maska][KOL_NOWY].tolist()

        if len(wyniki) < 4:
            print(f'  ⚠️  [{rok}] "{fraza}": '
                  f'{len(wyniki)} trafień zamiast 4 — NaN')
            dane[skrot] = [float('nan')] * 4
            continue

        vals = wyniki[:4]
        kw   = [vals[0], vals[1]-vals[0], vals[2]-vals[1], vals[3]-vals[2]]
        dane[skrot] = [
            v / 1_000_000 if pd.notna(v) else float('nan')
            for v in kw
        ]

    return dane


def parsuj_sprawozdanie(sciezka: str, rok: int) -> list:
    engine = 'openpyxl' if str(sciezka).endswith('.xlsx') else 'xlrd'

    df_raw = pd.read_excel(
        sciezka,
        sheet_name='TABLICA 3',
        engine=engine,
        header=None,
    )

    if wykryj_format_e(df_raw):
        dane = parsuj_format_e(df_raw, rok)
    elif rok in FORMAT_MAP:
        kol_nazwa, kol_suma = FORMAT_MAP[rok]
        dane = parsuj_bloki(df_raw, kol_nazwa, kol_suma, rok)
    else:
        print(f'  ❌ [{rok}] Nieznany format — pomijam')
        return []

    rekordy = []
    for q_idx, q_label in enumerate(['Q1', 'Q2', 'Q3', 'Q4']):
        kwartal = f'{rok}{q_label}'
        rekord  = {'kwartal': kwartal}
        for skrot in PODATKI_Q.values():
            rekord[f'{skrot}_mld'] = dane[skrot][q_idx]
        rekordy.append(rekord)
        print(f'  ✅ {kwartal}: '
              f'VAT={rekord["vat_mld"]:.3f} | '
              f'CIT={rekord["cit_mld"]:.3f} | '
              f'Akcyza={rekord["akcyza_mld"]:.3f} mld PLN')

    return rekordy


# ── Iteracja po latach ────────────────────────────────────────────────────────
LATA = range(1999, 2026)   # do 2025 włącznie

print('\n⏳ Parsowanie plików sprawozdań MF...')
print(f'📁 Katalog źródłowy: {DATA_RAW}')

rekordy_q = []
for rok in LATA:
    if rok == 2017:
        sciezka = DATA_RAW / 'sprawozdanie_operatywne_12_2017.XLS'
        if not sciezka.exists():
            sciezka = DATA_RAW / 'sprawozdanie_operatywne_12_2017.xls'
    elif rok >= 2018:
        sciezka = DATA_RAW / f'sprawozdanie_operatywne_12_{rok}.xlsx'
        if not sciezka.exists():
            sciezka = DATA_RAW / f'sprawozdanie_operatywne_12_{rok}.xls'
    else:
        sciezka = DATA_RAW / f'sprawozdanie_operatywne_12_{rok}.xls'

    if not sciezka.exists():
        print(f'  ⚠️  Brak pliku: {sciezka.name}')
        continue

    print(f'\n📄 Parsowanie: {sciezka.name}')
    try:
        rekordy_q.extend(parsuj_sprawozdanie(str(sciezka), rok))
    except Exception as e:
        print(f'  ❌ [{rok}] Błąd: {e}')

# ══════════════════════════════════════════════════════════════════════════════
# 1.3 Budowa panelu kwartalnego df
# ══════════════════════════════════════════════════════════════════════════════

df = (
    pd.DataFrame(rekordy_q)
    .set_index('kwartal')
    .sort_index()
)

# Zmienne pochodne
df['trend']      = range(1, len(df) + 1)
df['jpk_2016']   = (df.index >= '2016Q3').astype(int)
df['split_2018'] = (df.index >= '2018Q3').astype(int)
df['wlist_2019'] = (df.index >= '2019Q3').astype(int)
df['covid_2020'] = (df.index == '2020Q2').astype(int)

# Połączenie z PKB
df = df.join(pkb_q, how='left')
df['vat_pkb_pct'] = df['vat_mld'] / df['pkb_mld'] * 100

# ── Czyszczenie przestrzeni nazw ──────────────────────────────────────────────
del (data_pkb, time_labels, values_pkb, rows_pkb, pkb_q,
     response, url_pkb,
     idx, kwartal, value, rok, sciezka,
     rekordy_q, LATA, PODATKI_Q, KOL_NOWY,
     FORMAT_MAP, wykryj_format_e, parsuj_bloki,
     parsuj_format_e, parsuj_sprawozdanie)

# ══════════════════════════════════════════════════════════════════════════════
# 1.4 Agregacja roczna df_rok
# ══════════════════════════════════════════════════════════════════════════════

df.index = pd.PeriodIndex(df.index, freq='Q')

df_rok = df[['vat_mld', 'cit_mld', 'akcyza_mld', 'pkb_mld']].resample('Y').sum()
df_rok.index = df_rok.index.year

df_rok['vat_pkb_pct']    = df_rok['vat_mld'] / df_rok['pkb_mld'] * 100
df_rok['vat_udzial_pct'] = (
    df_rok['vat_mld'] /
    (df_rok['vat_mld'] + df_rok['cit_mld'] + df_rok['akcyza_mld']) * 100
)
df_rok['vat_rr']      = df_rok['vat_mld'].pct_change() * 100
df_rok['cit_akc_mld'] = df_rok['cit_mld'] + df_rok['akcyza_mld']
df_rok['trend']       = range(1, len(df_rok) + 1)

df.index = df.index.astype(str)

# ══════════════════════════════════════════════════════════════════════════════
# 1.5 Weryfikacja
# ══════════════════════════════════════════════════════════════════════════════

print(f'\n✅ ETL kwartalny zakończony')
print(f'   df:     {df.index.min()}–{df.index.max()}, n={len(df)}')
print(f'   df_rok: {df_rok.index.min()}–{df_rok.index.max()}, n={len(df_rok)}')
print(f'   Kolumny df: {df.columns.tolist()}')
print(f'   NaN VAT: {df["vat_mld"].isna().sum()}')
print()

print('=== WERYFIKACJA SUM ROCZNYCH ===')
for rok_check in [1999, 2003, 2016, 2017, 2024]:
    maska = df.index.str.startswith(str(rok_check))
    if maska.any():
        suma = df[maska]['vat_mld'].sum()
        print(f'  {rok_check}: VAT={suma:.3f} mld PLN')

# ══════════════════════════════════════════════════════════════════════════════
# 1.6 Zapis do parquet
# ══════════════════════════════════════════════════════════════════════════════

df.to_parquet(DATA_PROCESSED / 'df.parquet')
df_rok.to_parquet(DATA_PROCESSED / 'df_rok.parquet')

print(f'\n✅ Zapisano do: {DATA_PROCESSED}')
print(f'   - df.parquet  (kwartalny, n={len(df)})')
print(f'   - df_rok.parquet (roczny, n={len(df_rok)})')

# Czyszczenie
zachowaj = {'df', 'df_rok', 'gc', 'pd', 'np', 'Path',
            'DATA_PROCESSED', 'PROJECT_ROOT', 'DATA_RAW'}
for k in [k for k in list(globals().keys())
          if not k.startswith('_') and k not in zachowaj]:
    del globals()[k]
gc.collect()

print('\n✅ Notebook 01_etl wykonany pomyślnie!')

⏳ Pobieranie danych PKB z Eurostat API...
✅ PKB kwartalne: 1999Q1–2025Q4, n=108

⏳ Parsowanie plików sprawozdań MF...
📁 Katalog źródłowy: c:\Users\duros\OneDrive\Pulpit\Projekty Python\VAT\projekt_vat\raw

📄 Parsowanie: sprawozdanie_operatywne_12_1999.xls
  ✅ 1999Q1: VAT=11.289 | CIT=2.752 | Akcyza=5.244 mld PLN
  ✅ 1999Q2: VAT=11.874 | CIT=2.528 | Akcyza=6.127 mld PLN
  ✅ 1999Q3: VAT=12.114 | CIT=4.329 | Akcyza=7.066 mld PLN
  ✅ 1999Q4: VAT=13.534 | CIT=5.450 | Akcyza=6.770 mld PLN

📄 Parsowanie: sprawozdanie_operatywne_12_2000.xls
  ✅ 2000Q1: VAT=12.926 | CIT=3.763 | Akcyza=6.047 mld PLN
  ✅ 2000Q2: VAT=12.719 | CIT=2.628 | Akcyza=7.359 mld PLN
  ✅ 2000Q3: VAT=12.714 | CIT=4.885 | Akcyza=7.076 mld PLN
  ✅ 2000Q4: VAT=13.391 | CIT=5.591 | Akcyza=6.830 mld PLN

📄 Parsowanie: sprawozdanie_operatywne_12_2001.xls
  ✅ 2001Q1: VAT=12.589 | CIT=3.051 | Akcyza=6.240 mld PLN
  ✅ 2001Q2: VAT=12.716 | CIT=2.350 | Akcyza=7.134 mld PLN
  ✅ 2001Q3: VAT=13.274 | CIT=3.591 | Akcyza=7.670 mld PLN
  ✅ 